# Financial Indicators

screamer ships a rich set of financial technical indicators - RSI, MACD,
Bollinger Bands, ATR, and many more.  They all follow the same functor
interface, but indicators that consume multiple price series (High, Low,
Close) accept either *N positional arrays* or a single `(T, N)` matrix.
Multi-output indicators (MACD, Bollinger) return a `(T, N)` array where
the trailing axis indexes the outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import ATR, RollingRSI, MACD, BollingerBands

rng = np.random.default_rng(2)
close = 100 + np.cumsum(rng.standard_normal(400))
high  = close + np.abs(rng.standard_normal(400))
low   = close - np.abs(rng.standard_normal(400))

## Single-input indicators: RSI, MACD, Bollinger Bands

These indicators take a single price series (typically `close`) and return
either a 1-D series (RSI) or a `(T, N)` matrix where columns are the
indicator components.

- `BollingerBands(20)` → `(T, 3)`: columns are **lower**, **mid**, **upper**
- `MACD()` → `(T, 3)`: columns are **macd line**, **signal line**, **histogram**

In [ ]:
rsi  = RollingRSI(14)(close)
bb   = BollingerBands(20)(close)     # (400, 3): lower, mid, upper
macd = MACD()(close)                 # (400, 3): macd, signal, hist

print("RSI   shape:", rsi.shape)
print("BB    shape:", bb.shape,   " - columns: lower, mid, upper")
print("MACD  shape:", macd.shape, " - columns: macd, signal, hist")

# RSI stays in [0, 100]
assert np.nanmin(rsi) >= 0 and np.nanmax(rsi) <= 100
print("RSI in [0, 100] ✓")

## Multi-input OHLC: ATR and the `(T, N)` array form

`ATR` (Average True Range) needs High, Low, and Close.  You can pass them
as three separate 1-D arrays **or** stack them into a single `(T, 3)` matrix.
Both forms produce identical results.

In [ ]:
atr = ATR(14)(high, low, close)             # 3-input functor - 3 separate arrays

# the (T, N) form: pass one (T, 3) array instead of three args
hlc = np.column_stack([high, low, close])
np.testing.assert_array_equal(atr, ATR(14)(hlc))
print("ATR(high, low, close) == ATR(hlc) ✓")
print("ATR shape:", atr.shape)

## Price panel with indicator overlays

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(9, 7), sharex=True)

# price + Bollinger bands
ax[0].plot(close, lw=0.7, label="close")
ax[0].plot(bb[:, 0], "r--", lw=0.5, label="lower band")
ax[0].plot(bb[:, 2], "r--", lw=0.5, label="upper band")
ax[0].fill_between(range(len(close)), bb[:, 0], bb[:, 2], alpha=0.1, color="r")
ax[0].set_title("price + Bollinger Bands(20)")
ax[0].legend(loc="upper left", fontsize=8)

# RSI with overbought/oversold lines
ax[1].plot(rsi, label="RSI(14)")
ax[1].axhline(70, color="r", lw=0.4, ls="--", label="overbought")
ax[1].axhline(30, color="g", lw=0.4, ls="--", label="oversold")
ax[1].set_ylim(0, 100)
ax[1].set_title("RSI(14)")
ax[1].legend(loc="upper left", fontsize=8)

# ATR (volatility)
ax[2].plot(atr, label="ATR(14)")
ax[2].set_title("ATR(14) - rolling volatility")
ax[2].legend(loc="upper left", fontsize=8)

plt.tight_layout()

**Takeaways**

- All financial indicators are functors: `Indicator(params)(data)` →
  array or matrix.
- Multi-output indicators return `(T, N)`: trailing axis = output components.
- Multi-input indicators accept N separate 1-D arrays *or* one `(T, N)` matrix
  - use whichever form your data source produces.
- The same indicator works identically in backtest (array) and live (scalar)
  mode - no special streaming variant required.